# Notebook 1 of 2 — my remaining test pieces

You have printed the first part and measured its slot and edge. This rebuilds
the **seven remaining test pieces** at your printer's scale, and nothing else.

1. Paste your **round one** code below — it starts `S1U1-`.
2. Click **Runtime → Run all**.
3. Wait about five minutes; the download starts on its own.

It will not build the enclosure. Those come from **notebook 2**, after you have
printed and measured these seven.


In [ ]:
#@title Step 1 — paste your round one code here, then press the ▶ button { display-mode: "form" }
MY_CODE = ""  #@param {type:"string"}

import base64

# Two notebooks, two codes, and each refuses the other's. Getting this wrong is
# not a small mistake: a round-one code carries scale and nothing else, so
# building the enclosure from it would size every hole, seat and gasket gap to
# a value nobody has measured yet.
WANT = "S1U1-"
OTHER = "S1U-"
OTHER_NAME = "final"

KEYS = ["xy_scale_correction_fraction","z_scale_correction_fraction","fastener_clearance_diameter_offset_mm","insert_bore_diameter_offset_mm","driver_cutout_diameter_offset_mm","passive_radiator_cutout_diameter_offset_mm","cable_passage_diameter_offset_mm","gasket_sheet_thickness_mm","gasket_compressed_thickness_offset_mm","active_driver_flange_thickness_mm","passive_radiator_flange_thickness_mm"]

text = MY_CODE.strip()
if not text:
    raise SystemExit("Paste your code into the box above, then run this cell again.")
if text.startswith(OTHER):
    raise SystemExit(
        f"That is a {OTHER_NAME} code. This notebook wants one starting {WANT}.\n"
        "Open the other notebook, or go back to the website's Calibrate step.")
if not text.startswith(WANT):
    raise SystemExit(f"That does not look like a code. It should start with {WANT}")

payload = text[len(WANT):]
try:
    numbers = [float(v) for v in base64.b64decode(payload + "=" * (-len(payload) % 4)).decode().split(",")]
except Exception:
    raise SystemExit("That code looks damaged. Copy it again from the website.")
if len(numbers) != len(KEYS):
    raise SystemExit("That code is incomplete. Copy it again from the website.")

CALIBRATION = dict(zip(KEYS, numbers))

# Round one has two measurements in it. The other nine fields exist because the
# correction file has a fixed shape, and they carry the design's own defaults --
# printing all eleven under the heading "your measurements" reads as though the
# builder supplied figures they have not taken yet.
MEASURED = ["xy_scale_correction_fraction", "z_scale_correction_fraction"]

print("Read from your code:\n")
for key in MEASURED:
    percent = CALIBRATION[key] * 100.0
    axis = "XY" if key.startswith("xy") else "Z"
    if abs(percent) < 0.005:
        print(f"  {axis:2s} scale   your printer is dead on")
    else:
        bigger = "larger" if percent > 0 else "smaller"
        print(f"  {axis:2s} scale   {percent:+.2f}%  (parts will be built {abs(percent):.2f}% {bigger})")

print("\nNot measured yet, left at the design defaults until round two:")
for key, value in CALIBRATION.items():
    if key not in MEASURED:
        print(f"  {key:45s} {value}")


## Everything below runs on its own

You do not need to change anything here.


In [ ]:
#@title Install the CAD engine (about 3 minutes)
# numpy is held below 2.1 to match the numba that Colab pre-installs. Nothing
# here imports numba, so the mismatch was harmless, but pip printed it in red as
# an ERROR and that is not a thing to show someone halfway through their first
# print. cadquery declares no numpy requirement of its own, and the arrays used
# in this project are elementary, so the older line is equivalent for our
# purposes. If Colab's numba moves, this pin can move with it.
# This list is not a guess: it is every non-stdlib package reachable from the
# imports this notebook makes, and tests/test_notebook.py walks that import
# chain and fails if the two ever disagree. trimesh went missing when the
# editable install was removed, because that install was the only thing that
# had ever mentioned the project's dependencies, and the failure landed in the
# build cell rather than here.
!pip install --quiet cadquery==2.6.1 pyyaml==6.0.2 trimesh==4.6.10 "numpy<2.1" "scipy<1.16" 2>&1 | tail -2
print("CAD engine ready.")


In [ ]:
#@title Fetch the Satellite1 Ultra design
import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path("/content/Satellite1-Ultra")

# This used to run `pip install -e .`, which refuses on any Python outside the
# >=3.12,<3.13 pin in pyproject.toml. That pin describes the development and CI
# environment, not the source: every module here parses as 3.11. Colab moves its
# Python from time to time, so on the wrong day the install failed, the failure
# was piped through `tail -1` and hidden, and the next cell died with a bare
# "No module named 'satellite1_ultra'" that says nothing about the real cause.
#
# The package is pure Python, so putting src on the path is equivalent, cannot
# fail for a version reason, and needs no build backend.
shutil.rmtree(REPO, ignore_errors=True)
clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/BigPappy098/Satellite1-Ultra.git", str(REPO)],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    raise SystemExit("Could not download the design files:\n" + clone.stderr[-900:])

os.chdir(REPO)                       # the config file is written relative to here
sys.path.insert(0, str(REPO / "src"))

try:
    import satellite1_ultra  # noqa: F401
    from satellite1_ultra.configuration import ROOT
except Exception as error:
    raise SystemExit(
        f"The design files downloaded but would not load on Python "
        f"{sys.version.split()[0]}.\n{type(error).__name__}: {error}"
    )

assert ROOT == REPO, f"the package resolved its root to {ROOT}, not {REPO}"
print(f"Design files ready. Python {sys.version.split()[0]}.")


In [ ]:
#@title Check your numbers are safe
from satellite1_ultra.configuration import validate_physical_calibration

validate_physical_calibration(CALIBRATION)   # refuses anything physically implausible

with open("config/physical_calibration.yaml", "w") as handle:
    handle.write("# Generated from your measurements.\n")
    for key, value in CALIBRATION.items():
        handle.write(f"{key}: {value}\n")
print("Your numbers passed every safety check.")


In [ ]:
#@title Build your seven test pieces (about 2 minutes)
import time
from pathlib import Path
from satellite1_ultra.builder_files import CALIBRATION_STAGE_TWO
from satellite1_ultra.configuration import load_design_parameters
from satellite1_ultra.exporting import export_parts

# Only the coupons. Building the whole catalogue to hand back seven small parts
# would cost about ten minutes of your time for nothing.
WANTED = [source for source, _friendly, _quantity in CALIBRATION_STAGE_TWO
          if source != "coupon_official_interface"]

start = time.time()
parameters = load_design_parameters()
export_parts(Path("exports"), parameters, only=WANTED)
print(f"\nBuilt {len(WANTED)} test pieces in {(time.time()-start)/60:.1f} minutes.")


In [ ]:
#@title Package them and download
import shutil, os
from pathlib import Path
from satellite1_ultra.builder_files import CALIBRATION_STAGE_TWO

out = Path("/content/MY_TEST_PIECES")
shutil.rmtree(out, ignore_errors=True)
(out / "PRINT_THESE_SEVEN").mkdir(parents=True, exist_ok=True)

for source, friendly, _quantity in CALIBRATION_STAGE_TWO:
    if source == "coupon_official_interface":
        continue   # already printed and measured; that is where this code came from
    shutil.copy2(Path("exports/3mf") / f"{source}.3mf", out / "PRINT_THESE_SEVEN" / friendly)
    shutil.copy2(Path("exports/stl") / f"{source}.stl",
                 out / "PRINT_THESE_SEVEN" / f"{Path(friendly).stem}.stl")

(out / "READ_ME.txt").write_text(
    "Round two test pieces, sized for your printer.\n\n"
    "Print all seven, one at a time, with the same settings you used for the\n"
    "first part. The last one is the flexible cable seal, so load TPU.\n\n"
    "Do NOT sand or ream anything to fit. A piece that comes out wrong is the\n"
    "measurement we need.\n\n"
    "Then go back to the website's Calibrate step and fill in the\n"
    "'Measure those seven' tab. That gives you the final code, which goes into\n"
    "the SECOND notebook to make the real parts.\n")

archive = shutil.make_archive("/content/MY_TEST_PIECES", "zip", out.parent, out.name)
print(f"Ready: {os.path.getsize(archive)/1e6:.1f} MB")
try:
    from google.colab import files
    files.download(archive)
    print("\nYour download should start now.")
except Exception:
    print("\nOpen the folder icon on the left and download MY_TEST_PIECES.zip")


## Done

Your parts are in **MY_TEST_PIECES.zip**.

Print all seven, then go back to the website's **Calibrate** step and fill in
the *Measure those seven* tab. That gives you the final code for **notebook 2**.
